# Topic 3 – FastAPI in Practice ⚡
### From the paper contract (Topic 2) to a real, running game API

In Topic 1 we **consumed** APIs. In Topic 2 we **judged** their design. Now we **build our own**:
a *Game Power API* that predicts a character's power from its stats. The prediction is a simple
formula for now; in Topic 5 we replace it with a real trained linear regression model.

| Step | What we do |
|---|---|
| 1 | Smallest possible FastAPI app (hello world) |
| 2 | Path parameters and query parameters |
| 3 | Write the real app file `03_game_api.py` (the Topic 2 contract) |
| 4 | Call it **without a server** using `TestClient` |
| 5 | Automatic documentation (`/docs`, `/openapi.json`) |
| 6 | Start a **real server** and call it with `requests` (like Topic 1) |
| 7 | Test suite with PASS / FAIL (like Topic 2) |
| 8 | Run it from a terminal with `uvicorn --reload` |

## Step -1 – Install dependencies (run once)
Installs into **the kernel you selected**. If the next cell still says `ModuleNotFoundError`, restart the kernel and run again.

In [ ]:
%pip install -q fastapi uvicorn requests httpx

## Step 0 – Imports and a tiny test helper

In [1]:
import json
import sys
import os
import time
import threading
import importlib

import requests
import uvicorn
from fastapi import FastAPI
from fastapi.testclient import TestClient

RESULTS = []


def check(name, condition, detail=""):
    """Record and print one test case (same helper as Topic 2)."""
    RESULTS.append((name, bool(condition)))
    print(f"[{'PASS' if condition else 'FAIL'}] {name}" + (f"  -> {detail}" if detail else ""))


print("Working folder:", os.getcwd())

Working folder: C:\Users\LYampa\AppData\Local\Temp\claude\C--Users-LYampa-OneDrive---BOFI-Federal-Bank-Documents-My-Scripts-Python-PythonExercicesMachineLearning-main\1fc403ae-b66a-4e89-9b1e-df8355df465d\scratchpad


C:\Users\LYampa\AppData\Local\Temp\claude\C--Users-LYampa-OneDrive---BOFI-Federal-Bank-Documents-My-Scripts-Python-PythonExercicesMachineLearning-main\1fc403ae-b66a-4e89-9b1e-df8355df465d\scratchpad\libs\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## Step 1 – Hello World: the smallest API
Three ideas:
1. `app = FastAPI()` creates the application.
2. `@app.get("/")` links a **URL + method** to a Python function.
3. Whatever the function returns (a dict) is converted to **JSON** automatically.

`TestClient` lets us call the app directly from Python, with no server and no network.

In [2]:
hello_app = FastAPI(title="Hello API")


@hello_app.get("/")
def hello():
    return {"message": "Hello, player one!"}


client = TestClient(hello_app)
r = client.get("/")

print("Status      :", r.status_code)
print("Content-Type:", r.headers["content-type"])
print("Body        :", r.json())

Status      : 200
Content-Type: application/json
Body        : {'message': 'Hello, player one!'}


## Step 2 – Path parameters and query parameters
From Topic 2: the **path** identifies the resource, the **query** tweaks the result.
FastAPI reads both from the **function signature**; the type hints (`int`) validate them for free.

In [3]:
param_app = FastAPI()


@param_app.get("/characters/{character_id}")        # path parameter
def one_character(character_id: int, verbose: bool = False):   # query parameter with default
    data = {"id": character_id, "name": "pikachu"}
    if verbose:
        data["stats"] = {"attack": 55, "defense": 40, "speed": 90}
    return data


c = TestClient(param_app)
print(c.get("/characters/25").json())
print(c.get("/characters/25?verbose=true").json())

{'id': 25, 'name': 'pikachu'}
{'id': 25, 'name': 'pikachu', 'stats': {'attack': 55, 'defense': 40, 'speed': 90}}


In [4]:
# Type hints at work: "abc" is not an int
r = c.get("/characters/abc")
print("Status:", r.status_code)
print(json.dumps(r.json(), indent=2)[:400])

Status: 422
{
  "detail": [
    {
      "type": "int_parsing",
      "loc": [
        "path",
        "character_id"
      ],
      "msg": "Input should be a valid integer, unable to parse string as an integer",
      "input": "abc"
    }
  ]
}


## Step 3 – The real app: our Topic 2 contract in code
We write the file `03_game_api.py` with the notebook magic `%%writefile`. A real project keeps the app in a
`.py` file so a server can import it (Step 8).

Contract from Topic 2:

| Route | Purpose |
|---|---|
| `GET /v1/characters` | list characters (paginated) |
| `GET /v1/characters/{id}` | one character |
| `POST /v1/predictions` | predict power from stats |

In [5]:
%%writefile 03_game_api.py
"""Game Power API: Topic 3 of the ML + UI + API + MCP learning path."""
from fastapi import FastAPI, HTTPException, Query
from pydantic import BaseModel

app = FastAPI(
    title="Game Power API",
    version="1.0.0",
    description="Predicts a game character's power from its stats.",
)

# In-memory "database" (a real project would use a file or a DB)
CHARACTERS = {
    1: {"id": 1, "name": "pikachu", "attack": 55, "defense": 40, "speed": 90},
    2: {"id": 2, "name": "bulbasaur", "attack": 49, "defense": 49, "speed": 45},
    3: {"id": 3, "name": "charmander", "attack": 52, "defense": 43, "speed": 65},
    4: {"id": 4, "name": "squirtle", "attack": 48, "defense": 65, "speed": 43},
    5: {"id": 5, "name": "snorlax", "attack": 110, "defense": 65, "speed": 30},
    6: {"id": 6, "name": "gengar", "attack": 65, "defense": 60, "speed": 110},
    7: {"id": 7, "name": "mewtwo", "attack": 110, "defense": 90, "speed": 130},
}


class PredictionRequest(BaseModel):
    attack: int
    defense: int
    speed: int


class PredictionResponse(BaseModel):
    predicted_power: float
    model_version: str


@app.get("/")
def root():
    return {"message": "Game Power API is running", "docs": "/docs"}


@app.get("/v1/characters")
def list_characters(page: int = Query(1, ge=1), limit: int = Query(5, ge=1, le=50)):
    items = list(CHARACTERS.values())
    start = (page - 1) * limit
    return {
        "items": items[start:start + limit],
        "page": page,
        "limit": limit,
        "total": len(items),
    }


@app.get("/v1/characters/{character_id}")
def get_character(character_id: int):
    if character_id not in CHARACTERS:
        raise HTTPException(
            status_code=404,
            detail={"code": "CHARACTER_NOT_FOUND",
                    "message": f"Character {character_id} does not exist"},
        )
    return CHARACTERS[character_id]


@app.post("/v1/predictions", response_model=PredictionResponse)
def create_prediction(request: PredictionRequest):
    # Placeholder formula. In Topic 5 this becomes model.predict(...)
    power = 1.5 * request.attack + request.defense + 0.5 * request.speed
    return {"predicted_power": power, "model_version": "0.1.0"}

Writing 03_game_api.py


In [6]:
# Import the file we just wrote (its name starts with a digit, so we use importlib)
sys.path.insert(0, os.getcwd())
game_api = importlib.import_module("03_game_api")
importlib.reload(game_api)          # pick up changes if you edit the file and re-run

app = game_api.app
print("Routes registered:")
for route in app.routes:
    methods = ",".join(sorted(getattr(route, "methods", []) or []))
    print(f"  {methods:<10} {route.path}")

Routes registered:
  GET,HEAD   /openapi.json
  GET,HEAD   /docs
  GET,HEAD   /docs/oauth2-redirect
  GET,HEAD   /redoc
  GET        /
  GET        /v1/characters
  GET        /v1/characters/{character_id}
  POST       /v1/predictions


## Step 4 – Calling the app with `TestClient` (no server)
Same requests as Topic 1, but straight into our code. This is the fastest way to test an API.

In [7]:
client = TestClient(app)

print("GET /v1/characters?limit=3")
print(json.dumps(client.get("/v1/characters", params={"limit": 3}).json(), indent=2)[:500])

GET /v1/characters?limit=3
{
  "items": [
    {
      "id": 1,
      "name": "pikachu",
      "attack": 55,
      "defense": 40,
      "speed": 90
    },
    {
      "id": 2,
      "name": "bulbasaur",
      "attack": 49,
      "defense": 49,
      "speed": 45
    },
    {
      "id": 3,
      "name": "charmander",
      "attack": 52,
      "defense": 43,
      "speed": 65
    }
  ],
  "page": 1,
  "limit": 3,
  "total": 7
}


In [8]:
r = client.get("/v1/characters/7")
print(r.status_code, r.json())

r = client.get("/v1/characters/99")
print(r.status_code, r.json())          # our standard error shape (inside "detail")

200 {'id': 7, 'name': 'mewtwo', 'attack': 110, 'defense': 90, 'speed': 130}
404 {'detail': {'code': 'CHARACTER_NOT_FOUND', 'message': 'Character 99 does not exist'}}


In [9]:
payload = {"attack": 55, "defense": 40, "speed": 90}
r = client.post("/v1/predictions", json=payload)
print(r.status_code, r.json())

200 {'predicted_power': 167.5, 'model_version': '0.1.0'}


In [10]:
# Bad input: text instead of a number. FastAPI answers 422 automatically
r = client.post("/v1/predictions", json={"attack": "high", "defense": 40, "speed": 90})
print(r.status_code)
print(json.dumps(r.json(), indent=2))

422
{
  "detail": [
    {
      "type": "int_parsing",
      "loc": [
        "body",
        "attack"
      ],
      "msg": "Input should be a valid integer, unable to parse string as an integer",
      "input": "high"
    }
  ]
}


## Step 5 – Free documentation
FastAPI builds an **OpenAPI** description of every route from our code. Two pages come for free once a server runs:

- `/docs` → Swagger UI (try requests in the browser)
- `/redoc` → readable reference
- `/openapi.json` → the machine-readable contract (later, MCP tools can be built from the same idea)

In [11]:
spec = client.get("/openapi.json").json()
print("Title  :", spec["info"]["title"], spec["info"]["version"])
print("Paths  :")
for path, methods in spec["paths"].items():
    print(f"  {path:<30} {', '.join(m.upper() for m in methods)}")

print()
print("Schemas:", list(spec["components"]["schemas"].keys()))
print(json.dumps(spec["components"]["schemas"]["PredictionRequest"], indent=2))

Title  :

 Game Power API 1.0.0
Paths  :
  /                              GET
  /v1/characters                 GET
  /v1/characters/{character_id}  GET
  /v1/predictions                POST

Schemas: ['HTTPValidationError', 'PredictionRequest', 'PredictionResponse', 'ValidationError']
{
  "properties": {
    "attack": {
      "type": "integer",
      "title": "Attack"
    },
    "defense": {
      "type": "integer",
      "title": "Defense"
    },
    "speed": {
      "type": "integer",
      "title": "Speed"
    }
  },
  "type": "object",
  "required": [
    "attack",
    "defense",
    "speed"
  ],
  "title": "PredictionRequest"
}


## Step 6 – A real server, called with `requests`
`uvicorn` is the program that listens on a **port** and passes HTTP requests to our app.
We start it in a background thread so this notebook keeps working, then call it exactly like in Topic 1.

In [12]:
PORT = 8765
BASE = f"http://127.0.0.1:{PORT}"

config = uvicorn.Config(app, host="127.0.0.1", port=PORT, log_level="warning")
server = uvicorn.Server(config)
thread = threading.Thread(target=server.run, daemon=True)
thread.start()

# wait until the server answers
for _ in range(50):
    try:
        requests.get(BASE, timeout=0.5)
        break
    except requests.exceptions.ConnectionError:
        time.sleep(0.1)

print("Server is up at", BASE)
print("Open in your browser:", BASE + "/docs")

Server is up at http://127.0.0.1:8765
Open in your browser: http://127.0.0.1:8765/docs


In [13]:
r = requests.get(f"{BASE}/v1/characters/7", timeout=10)
print("Status :", r.status_code)
print("Server :", r.headers.get("server"))
print("Body   :", r.json())

r = requests.post(f"{BASE}/v1/predictions",
                  json={"attack": 110, "defense": 90, "speed": 130}, timeout=10)
print()
print("Prediction for mewtwo-like stats:", r.json())

Status : 200
Server : uvicorn
Body   : {'id': 7, 'name': 'mewtwo', 'attack': 110, 'defense': 90, 'speed': 130}



Prediction for mewtwo-like stats: {'predicted_power': 320.0, 'model_version': '0.1.0'}


## Step 7 – Test suite (PASS / FAIL)
Topic 2 rules, now checked against **our own** API.

In [14]:
# Contract: routes are versioned plural nouns
paths = list(client.get("/openapi.json").json()["paths"])
check("All business routes start with /v1", all(p.startswith("/v1/") for p in paths if p != "/"))

# Happy paths
check("list -> 200", client.get("/v1/characters").status_code == 200)
body = client.get("/v1/characters", params={"page": 1, "limit": 3}).json()
check("list returns the pagination envelope", {"items", "page", "limit", "total"} <= set(body))
check("page size respected", len(body["items"]) == 3)
check("total is the full size", body["total"] == len(game_api.CHARACTERS))

# Pagination edges
check("page past the end is empty", client.get("/v1/characters", params={"page": 99}).json()["items"] == [])
check("limit=0 rejected (422)", client.get("/v1/characters", params={"limit": 0}).status_code == 422)
check("limit=500 rejected (422)", client.get("/v1/characters", params={"limit": 500}).status_code == 422)

# One resource + errors
check("character 1 -> 200", client.get("/v1/characters/1").status_code == 200)
r = client.get("/v1/characters/99")
check("missing character -> 404", r.status_code == 404)
check("404 uses the standard error shape", set(r.json()["detail"]) == {"code", "message"})
check("id 'abc' -> 422", client.get("/v1/characters/abc").status_code == 422)

# Predictions
r = client.post("/v1/predictions", json={"attack": 55, "defense": 40, "speed": 90})
check("prediction -> 200", r.status_code == 200)
check("prediction has the contract fields", set(r.json()) == {"predicted_power", "model_version"})
check("formula result is correct", r.json()["predicted_power"] == 1.5 * 55 + 40 + 0.5 * 90)
check("missing field -> 422", client.post("/v1/predictions", json={"attack": 1}).status_code == 422)
check("text instead of number -> 422",
      client.post("/v1/predictions", json={"attack": "x", "defense": 1, "speed": 1}).status_code == 422)
check("GET on a POST route -> 405", client.get("/v1/predictions").status_code == 405)

# Same checks through the real server
check("real server answers 200", requests.get(f"{BASE}/v1/characters", timeout=10).status_code == 200)
check("real server runs on uvicorn", requests.get(BASE, timeout=10).headers.get("server") == "uvicorn")

passed = sum(ok for _, ok in RESULTS)
print(f"\n{passed}/{len(RESULTS)} test cases passed")

[PASS] All business routes start with /v1
[PASS] list -> 200
[PASS] list returns the pagination envelope
[PASS] page size respected
[PASS] total is the full size
[PASS] page past the end is empty
[PASS] limit=0 rejected (422)
[PASS] limit=500 rejected (422)
[PASS] character 1 -> 200
[PASS] missing character -> 404
[PASS] 404 uses the standard error shape
[PASS] id 'abc' -> 422
[PASS] prediction -> 200
[PASS] prediction has the contract fields
[PASS] formula result is correct
[PASS] missing field -> 422
[PASS] text instead of number -> 422
[PASS] GET on a POST route -> 405


[PASS] real server answers 200


[PASS] real server runs on uvicorn

20/20 test cases passed


In [15]:
# Stop the background server when we are done
server.should_exit = True
thread.join(timeout=5)
print("Server stopped:", not thread.is_alive())

Server stopped:

 True


## Step 8 – Run it like a real project (terminal)
Open a terminal **in this folder** and run:

```bash
uvicorn 03_game_api:app --reload --port 8000
```

- `03_game_api` → the Python file (without `.py`)
- `app` → the variable inside the file
- `--reload` → restarts the server every time you save the file (great for learning)

Then open <http://127.0.0.1:8000/docs>, click **POST /v1/predictions → Try it out → Execute**.

Common problems:

| Symptom | Fix |
|---|---|
| `uvicorn is not recognized` | use `python -m uvicorn 03_game_api:app --reload` |
| `Address already in use` | change the port: `--port 8001` |
| `Could not import module` | run the command from the folder that contains `03_game_api.py` |

### Exercises
1. Add `GET /v1/characters/{id}/power` that returns the formula result for a stored character.
2. Add a query parameter `name` to `GET /v1/characters` that filters by part of the name.
3. Add a new character and verify with a test that `total` increases.
4. Open `/docs` and send an invalid body. Read the 422 error: which field does it point to?

**Next:** Topic 4 – Pydantic. We make the input rules richer (ranges, required vs optional, custom messages) so the model only ever receives valid features.